In [1]:
import pandas as pd
from aksharamukha.transliterate import process
import re

In [2]:
# -------------------------------
# Custom Character Mappings
# -------------------------------

kashmiri_map = {
    # Vowels
    'ا': 'a', 'آ': 'ā', 'و': 'u', 'ؤ': 'ū', 'ی': 'i', 'ئ': 'i', 'ے': 'e', 'ۄ': 'o',
    'ہِ': 'hi', 'ہ': 'h', 'ہٕ': 'h̤', 'ہُ': 'hu', 'ہہ': 'hh', 'ہَ': 'ha', 'ٮ۪': 'n',

    # Native consonants
    'ب': 'b', 'پ': 'p', 'ت': 't', 'ٹ': 'ṭ', 'ج': 'j', 'چ': 'ch', 'ح': 'ḥ', 'خ': 'kh',
    'د': 'd', 'ڈ': 'ḍ', 'ر': 'r', 'ڑ': 'ṛ', 'ز': 'z', 'ژ': 'zh', 'س': 's', 'ش': 'sh',
    'ل': 'l', 'م': 'm', 'ن': 'n', 'ں': 'ṅ', 'ھ': 'h', 'ک': 'k', 'گ': 'g', 'ڳ': 'ġ',

    # Aspirated digraphs
    'تھ': 'th', 'دھ': 'dh', 'کھ': 'kh', 'گھ': 'gh', 'چھ': 'chh', 'جھ': 'jh',

    # Loan letters from Persian/Arabic/Urdu
    'ث': 's', 'ذ': 'z', 'ص': 'ṣ', 'ض': 'ẓ', 'ط': 'ṭ', 'ظ': 'ẓ', 'ع': '‘', 'غ': 'gh',
    'ف': 'f', 'ق': 'q', 'ڤ': 'v',

    # Special / other characters
    'ء': 'ʾ', 'ٔ': 'ʲ', '۔': '.', '،': ',', 'ہٕ': 'ḥ', 'ہِ': 'hi',
    'ك': 'k', 'ٲ': 'a', 'كہ': 'kah', 'ہكہِ': 'hakhi', 'ۍ': 'y', 'ہہ': 'hh', 'ٮ۪': 'q̇',
}

santhali_map = {
    '᱐': '0', '᱑': '1', '᱒': '2', '᱓': '3', '᱔': '4', '᱕': '5', '᱖': '6', '᱗': '7', '᱘': '8', '᱙': '9',
    'ᱚ': 'a', 'ᱛ': 'b', 'ᱜ': 'c', 'ᱝ': 'd', 'ᱞ': 'e', 'ᱟ': 'f', 'ᱠ': 'g', 'ᱡ': 'h',
    'ᱢ': 'i', 'ᱣ': 'j', 'ᱤ': 'k', 'ᱥ': 'l', 'ᱦ': 'm', 'ᱧ': 'n', 'ᱨ': 'o', 'ᱩ': 'p',
    'ᱪ': 'q', 'ᱫ': 'r', 'ᱬ': 's', 'ᱭ': 't', 'ᱮ': 'u', 'ᱯ': 'v', 'ᱰ': 'w', 'ᱱ': 'x',
    'ᱲ': 'y', 'ᱳ': 'z', 'ᱴ': 'ṭ', 'ᱵ': 'ḍ', 'ᱶ': 'ṅ', 'ᱷ': 'ṭh', 'ᱸ': 'ḍh', 'ᱹ': 'ñ',
    'ᱺ': 'ŋ', 'ᱻ': 'ś', 'ᱼ': 'ṣ', 'ᱽ': 'ḷ'
}

sindhi_map = {
    # Standard Arabic letters
    '۽': 'aṇ', 'ڊ': 'ḍ', 'ڪ': 'k', 'ڙ': 'ṛ', 'ڌ': 'dh',
    'ا': 'a', 'آ': 'ā', 'ب': 'b', 'ت': 't', 'ث': 's', 'ج': 'j',
    'ح': 'ḥ', 'خ': 'kh', 'د': 'd', 'ذ': 'z', 'ر': 'r',
    'ز': 'z', 'س': 's', 'ش': 'sh', 'ص': 'ṣ', 'ض': 'ẓ',
    'ط': 'ṭ', 'ظ': 'ẓ', 'ع': '‘', 'غ': 'gh', 'ف': 'f',
    'ق': 'q', 'ك': 'k', 'ک': 'k', 'گ': 'g', 'ل': 'l', 'م': 'm',
    'ن': 'n', 'ه': 'h', 'ھ': 'h', 'ء': 'ʾ', 'ي': 'y', 'ی': 'y',
    'و': 'w', 'ؤ': 'ʾw', 'إ': 'ʾi', 'أ': 'ʾa', 'ئ': 'ʾi',

    # Sindhi-specific consonants
    'پ': 'p', 'ٿ': 'th', 'ٹ': 'ṭ', 'ٽ': 'ṭ', 'ڇ': 'chh',
    'چ': 'ch', 'ڈ': 'ḍ', 'ڍ': 'ḍh', 'ڦ': 'ph', 'ڱ': 'ṅ',
    'ڻ': 'ṇ', 'ڽ': 'ñ', 'ڑ': 'ṛ', 'ڰ': 'ṛ', 'ڳ': 'ġ',
    'ڄ': 'jj', 'ژ': 'zh', 'ٻ': 'b', 'گw': 'gw', 'آṅ': 'āṅ',

    # Aspirated / multi-character combinations
    'ڍh': 'ḍh', 'ٺh': 'ṭh', 'ڻh': 'ṇh', 'ڱh': 'ṅh',

    # Vowels (letters and diacritics)
    'ے': 'e', 'ۓ': 'ē', 'ۄ': 'o', 'َ': 'a', 'ِ': 'i', 'ُ': 'u',
    'ٖ': 'e', 'ٰ': 'ā', 'ّ': '', 'ً': 'an', 'ٍ': 'in', 'ٌ': 'un',
    'ى': 'ā', 'آṅ': 'āṅ',

    # Nasalization / special marks
    '؎': '~', '۾': 'm',

    # Punctuation / other signs
    '،': ',', '۔': '.', '؛': ';', '؟': '?',
    'ڀ': 'bh', 'ڏ': 'dh', 'ٺ': 'ṭh', 'ʾ': 'ʾ'
}

urdu_map = {
    # Vowels / long vowels
    'ہ': 'h', 'ۂ': 'hʾ',
    'ا': 'a', 'آ': 'ā', 'و': 'ū', 'ۄ': 'o', 'ی': 'ī', 'ے': 'e', 'ۓ': 'ē',

    # Basic consonants
    'ب': 'b', 'پ': 'p', 'ت': 't', 'ٹ': 'ṭ', 'ث': 's', 'ج': 'j', 'چ': 'ch', 'ح': 'ḥ',
    'خ': 'kh', 'د': 'd', 'ڈ': 'ḍ', 'ذ': 'z', 'ر': 'r', 'ڑ': 'ṛ', 'ز': 'z', 'ژ': 'zh',
    'س': 's', 'ش': 'sh', 'ص': 'ṣ', 'ض': 'ẓ', 'ط': 'ṭ', 'ظ': 'ẓ', 'ع': '‘', 'غ': 'gh',
    'ف': 'f', 'ق': 'q', 'ک': 'k', 'گ': 'g', 'ل': 'l', 'م': 'm', 'ن': 'n', 'ں': 'ṅ',
    'ھ': 'h', 'و': 'w', 'ء': 'ʾ', 'ؤ': 'ʾ', 'ی': 'y', 'ئ': 'i',

    # Additional letters for loanwords / Sanskrit sounds
    'ڑ': 'ṛ', 'ڤ': 'v', 'ٹ': 'ṭ', 'ݨ': 'ṇ', 'ڱ': 'ṅ', 'ݙ': 'ḍ', 'ݚ': 'ṛ',
    'ݟ': 'ṭ', 'ݡ': 'ḍ', 'ݢ': 'g', 'ݣ': 'ġ', 'ݤ': 'dh', 'ݥ': 'bh',

    # Diacritics (used for vowel sounds)
    'َ': 'a', 'ِ': 'i', 'ُ': 'u', 'ّ': '', 'ً': 'an', 'ٍ': 'in', 'ٌ': 'un', 'ٰ': 'ā',

    # Punctuation
    '،': ',', '۔': '.'
}

# -------------------------------
# Mapping Dictionary
# -------------------------------

custom_mappings = {
    'kas_Arab': kashmiri_map,
    'snd_Arab': sindhi_map,
    'urd_Arab': urdu_map,
    'sat_Olck': santhali_map,
}

In [3]:
# -------------------------------
# Language → Aksharamukha script mapping
# -------------------------------
language_to_script = {
    'asm_Beng': 'Assamese',
    'ben_Beng': 'Bengali',
    'brx_Deva': 'Devanagari',
    'doi_Deva': 'Devanagari',
    'gom_Deva': 'Devanagari',
    'guj_Gujr': 'Gujarati',
    'hin_Deva': 'Devanagari',
    'kan_Knda': 'Kannada',
    # 'kas_Arab': 'Arabic',        # Kashmiri in Arabic script
    # 'kas_Deva': 'Devanagari',
    'mai_Deva': 'Devanagari',
    'mal_Mlym': 'Malayalam',
    'mar_Deva': 'Devanagari',
    # 'mni_Beng': 'Bengali',
    'mni_Mtei': 'MeeteiMayek',
    'npi_Deva': 'Devanagari',
    'ory_Orya': 'Oriya',
    'pan_Guru': 'Gurmukhi',
    'san_Deva': 'Devanagari',
    'snd_Deva': 'Devanagari',    # Sindhi in Devanagari version
    'tam_Taml': 'Tamil',
    'tel_Telu': 'Telugu',
    # 'urd_Arab': 'Arabic'         # Urdu in Arabic script
}


In [4]:
# -------------------------------
# Transliteration function
# -------------------------------
def transliterate_row(row):
    lang = row['tgt_lang']
    sentence = str(row['tgt'])
    
    # If custom mapping language
    if lang in custom_mappings:
        mapping = custom_mappings[lang]
        return ''.join([mapping.get(c, c) for c in sentence])
    
    # Use Aksharamukha for all other languages
    source_script = language_to_script.get(lang)
    target_script = 'IAST'  # Roman
    if not source_script:
        return '[Unknown language]'
    try:
        return process(source_script, target_script, sentence)
    except Exception as e:
        return f"[Error: {e}]"

In [5]:
def transliterate_row2(row):
    lang = row['tgt_lang']
    sentence = str(row['Roman2'])
    if lang in custom_mappings:
        mapping = custom_mappings[lang]
        return ''.join([mapping.get(c, c) for c in sentence])
    source_script = language_to_script.get(lang)
    target_script = 'IAST'
    if not source_script:
        return '[Unknown language]'
    try:
        return process(source_script, target_script, sentence)
    except Exception as e:
        return f"[Error: {e}]"

In [6]:
# snd_Arab = pd.concat([df[:816], df[3110:5097]], axis = 0, ignore_index=True)
# snd_Arab.to_csv("bpcc/langs_bpcc/snd_Arab.tsv", sep='\t', index=False)

# snd_Deva = pd.concat([df[816: 3110], df[5097:]], axis = 0, ignore_index=True)
# snd_Deva.to_csv("bpcc/langs_bpcc/snd_Deva.tsv", sep='\t', index=False)

In [7]:
# ==================== Adding Santali from Bhasha-Abhijnaanam training dataset ====================

# native = pd.read_csv("abhijnaanam_native.csv")
# native[native['label']=='Santali'].to_csv("bpcc/langs_bpcc/sat_Olck.tsv", sep='\t', index=False)

In [40]:
# sat = pd.read_csv("bpcc/langs_bpcc/sat_Olck.tsv", sep='\t')
# sat.head()

In [22]:
# sat = sat.rename(columns = {'label': 'tgt_lang', 'text': 'tgt'})
# sat.head()

In [16]:
# sat['label'] = 'sat_Olck'
# sat.head()

In [24]:
# sat.to_csv("bpcc/langs_bpcc/sat_Olck.tsv", sep='\t', index=False)

In [8]:
# -------------------------------
# Apply transliteration
# -------------------------------
for key, values in language_to_script.items():
    df = pd.read_csv(f"bpcc/langs_bpcc/{key}.tsv", sep="\t")
    df['Roman'] = df.apply(transliterate_row, axis=1)
    df.to_csv(f"bpcc/transliterated_bpcc/{key}.csv", index=False, encoding='utf-8-sig')

In [25]:
for key, values in custom_mappings.items():
    df = pd.read_csv(f"bpcc/langs_bpcc/{key}.tsv", sep="\t")
    df['Roman2'] = df.apply(transliterate_row, axis=1)
    df['Roman'] = df.apply(transliterate_row2, axis=1)
    df.drop('Roman2', axis=1, inplace=True)
    # if key == 'kas_Arab':
    df['Roman'] = df['Roman'].apply(lambda x: re.sub(r'[^A-Za-z0-9\s.,!?\'"-]', 'n', str(x)))
    df.to_csv(f"bpcc/transliterated_bpcc/{key}.csv", index=False, encoding='utf-8-sig')

In [26]:
for key, values in language_to_script.items():
    df = pd.read_csv(f"bpcc/transliterated_bpcc/{key}.csv")
    print(len(df))

99581
108000
99395
90158
72766
105850
96377
97175
98382
98178
117403
78204
120949
90440
73115
97646
16054
98038
98262


In [27]:
for key, values in custom_mappings.items():
    df = pd.read_csv(f"bpcc/transliterated_bpcc/{key}.csv")
    print(len(df))

88381
2803
99118
100345


In [35]:
# Bodo (brx)
# Dogri (doi)
# Gujarati (guj)
# Hindi (hin)
# Kannada (kan)
# Kashmiri (kas)
# Konkani (gom)
# Maithili (mai)
# Malayalam (mal)
# Marathi (mar)
# Manipuri / Meitei (mni)
# Nepali (npi)
# Odia (ori)
# Punjabi (pan)
# Sanskrit (san)
# Santali (sat)   not present in bpcc-seed-latest, adding santali native from bhasha-abhijnaanam training dataset
# Sindhi (snd)
# Tamil (tam)
# Telugu (tel)
# Urdu (urd)

In [36]:
len(custom_mappings) + len(language_to_script)

23

In [37]:
dfs = []

# Read all transliterated CSVs
for key, value in custom_mappings.items():
    df = pd.read_csv(f"bpcc/transliterated_bpcc/{key}.csv")
    dfs.append(df)
    
for key, value in language_to_script.items():
    df = pd.read_csv(f"bpcc/transliterated_bpcc/{key}.csv")
    dfs.append(df)

# Concatenate all DataFrames vertically and reset index
df_combined = pd.concat(dfs, axis=0, ignore_index=True)
df_combined

,src_lang,tgt_lang,src,tgt,Roman
0,eng_Latn,kas_Arab,"He bickers with the maids, harrows his hapless...",سُہ چُھ داین سۭتۍ لَڑٲے کَران، پنٛنِس مُصیٖبت ...,"snh chnh dain snty lnnae knran, pnnnns mnninbt..."
1,eng_Latn,kas_Arab,"The Nizam, seeking to deflect the Company from...",نظامَن اوس حیدر علی یَس سٕتہِ ،کمپنی شٗمٲلی س...,"nnamnn aus nidr nli ins snthn ,kmpni shnmali ..."
2,eng_Latn,kas_Arab,In 1766 Mysore began to become drawn into terr...,1766 منٛز ہیو٘تٗن میسور حیدرآبادٕ کِس نظامس ت...,1766 mnnz hiuntnn misur nidrnbadn kns nnams t...
3,eng_Latn,kas_Arab,"Ray began to make illustrations for it, as wel...",رے ین خمرِ امہِ باپت تصویر تیار كِرٕنہِ ،تہِ...,"re in khmrn amhn bapt tnuir tiar knrnnhn ,th..."
4,eng_Latn,kas_Arab,"Prithviraj, who was a minor at the time, ascen...",پرتھوی راج یُس تَمہِ وِزِ لو٘كٗٹ اوس، بِیوٗٹھ...,"prthui raj ins tnmhn unzn lunknn aus, bniunnh..."
...,...,...,...,...,...
2046615,eng_Latn,tel_Telu,"On August 11, 2020, Biden picked Harris as his...",బైడెన్ 2020 ఆగస్టు 11న హారిస్‌ను తన సహచరుడిగా ...,baiḍĕn 2020 āgasṭu 11na hārisnu tana sahacaruḍ...
2046616,eng_Latn,tel_Telu,Ivey served as the 38th Alabama State Treasure...,ఇవే 2003 నుండి 2011 వరకు 38వ అలబామా రాష్ట్ర కో...,ive 2003 nuṃḍi 2011 varaku 38va alabāmā rāṣṭra...
2046617,eng_Latn,tel_Telu,Julia Gillard was elected to the position of l...,"జూలియా గిల్లార్డ్ నాయకురాలిగా ఎన్నికయ్యారు, అం...","jūliyā gillārḍ nāyakurāligā ĕnnikayyāru, aṃduv..."
2046618,eng_Latn,tel_Telu,It swelled up to the size of an orange at its ...,అది దాని అత్యధిక స్థాయిలో నారింజ పరిమాణానికి ప...,adi dāni atyadhika sthāyilo nāriṃja parimāṇāni...


In [38]:
df_combined.to_csv('transliterated_bpcc.csv', index=False)

In [44]:
df_combined['tgt_lang'].value_counts()

tgt_lang
npi_Deva    120949
mar_Deva    117403
ben_Beng    108000
guj_Gujr    105850
sat_Olck    100345
asm_Beng     99581
brx_Deva     99395
urd_Arab     99118
mai_Deva     98382
tel_Telu     98262
mal_Mlym     98178
tam_Taml     98038
san_Deva     97646
kan_Knda     97175
hin_Deva     96377
ory_Orya     90440
doi_Deva     90158
kas_Arab     88381
mni_Mtei     78204
pan_Guru     73115
gom_Deva     72766
snd_Deva     18857
Name: count, dtype: int64

## Adding Dogri transliterated sentences to get 100k sentences for each class

In [41]:
native = pd.read_csv("abhijnaanam_native.csv")
native[native['label']=='Dogri'].to_csv("bpcc/langs_bpcc/dogri.tsv", sep='\t', index=False)

In [47]:
df = pd.read_csv("bpcc/langs_bpcc/dogri.tsv", sep="\t")
df = df.rename(columns = {'label': 'tgt_lang', 'text': 'tgt'})
df['tgt_lang'] = 'doi_Deva'
df['Roman'] = df.apply(transliterate_row, axis=1)
df.to_csv(f"bpcc/transliterated_bpcc/dogri.csv", index=False, encoding='utf-8-sig')

In [54]:
combined = pd.read_csv("transliterated_bpcc.csv")
combined.head()

,src_lang,tgt_lang,src,tgt,Roman
0,eng_Latn,kas_Arab,"He bickers with the maids, harrows his hapless...",سُہ چُھ داین سۭتۍ لَڑٲے کَران، پنٛنِس مُصیٖبت ...,"snh chnh dain snty lnnae knran, pnnnns mnninbt..."
1,eng_Latn,kas_Arab,"The Nizam, seeking to deflect the Company from...",نظامَن اوس حیدر علی یَس سٕتہِ ،کمپنی شٗمٲلی س...,"nnamnn aus nidr nli ins snthn ,kmpni shnmali ..."
2,eng_Latn,kas_Arab,In 1766 Mysore began to become drawn into terr...,1766 منٛز ہیو٘تٗن میسور حیدرآبادٕ کِس نظامس ت...,1766 mnnz hiuntnn misur nidrnbadn kns nnams t...
3,eng_Latn,kas_Arab,"Ray began to make illustrations for it, as wel...",رے ین خمرِ امہِ باپت تصویر تیار كِرٕنہِ ،تہِ...,"re in khmrn amhn bapt tnuir tiar knrnnhn ,th..."
4,eng_Latn,kas_Arab,"Prithviraj, who was a minor at the time, ascen...",پرتھوی راج یُس تَمہِ وِزِ لو٘كٗٹ اوس، بِیوٗٹھ...,"prthui raj ins tnmhn unzn lunknn aus, bniunnh..."


In [52]:
filtered_df = df[df['Roman'].str.split().apply(len).between(10, 50)]
filtered_df

,tgt_lang,tgt,Roman
1,doi_Deva,ते रवि कोला खोई गेआ जे अपनी सेह्तु दा ध्यान आप...,te ravi kolā khoī geā je apanī sehtu dā dhyāna...
4,doi_Deva,इ ʼ यां ' साइबर - एरा ' संयुक्त शब्द थमां अभिप...,i yāṃ ' sāibara - erā ' saṃyukta śabda thamāṃ...
6,doi_Deva,राम नाथ स्मारक समिति आसेआ गवर्नमेंट गर्ल्स हाय...,rāma nātha smāraka samiti āseā gavarnameṃṭa ga...
8,doi_Deva,जेकर तुसेंगी जकीन होए जे केंदर लोक सूचना अधिका...,jekara tuseṃgī jakīna hoe je keṃdara loka sūca...
12,doi_Deva,ओह्दे जीवनै दी गति रूकी दी गै लगदी ऎ ।,ohde jīvanai dī gati rūkī dī gai lagadī ĕ .
...,...,...,...
100115,doi_Deva,इʼनें सभनें जुगतियें गी सोके दे दुरान इमानदारी...,ineṃ sabhaneṃ jugatiyeṃ gī soke de durāna imān...
100116,doi_Deva,2016 - 17 आस्तै राश्ट्री राजधानी खेतर दिल्...,2016 - 17 āstai rāśṭrī rājadhānī khetara d...
100117,doi_Deva,जिʼयां गै दʼऊं परमाणु शक्ति आह्‌ले देशें दे ...,jiyāṃ gai döūṃ paramāṇu śakti āhle deśeṃ de ...
100118,doi_Deva,एह्‌‌‌दे स्वाय/अलावा उʼनैं पारिवारिक खताब श्री...,ehde svāya/alāvā ünaiṃ pārivārika khatāba śrī ...


In [55]:
combined = pd.concat([combined, filtered_df], axis=0, ignore_index=True)
combined.tgt_lang.value_counts()

tgt_lang
doi_Deva    134100
npi_Deva    120949
mar_Deva    117403
ben_Beng    108000
guj_Gujr    105850
sat_Olck    100345
asm_Beng     99581
brx_Deva     99395
urd_Arab     99118
mai_Deva     98382
tel_Telu     98262
mal_Mlym     98178
tam_Taml     98038
san_Deva     97646
kan_Knda     97175
hin_Deva     96377
ory_Orya     90440
kas_Arab     88381
mni_Mtei     78204
pan_Guru     73115
gom_Deva     72766
snd_Deva     18857
Name: count, dtype: int64

In [56]:
combined.to_csv('transliterated_bpcc.csv', index=False)